# Confluence Documentation Generator

This notebook generates comprehensive, Confluence-ready documentation with Mermaid diagrams.
It supports both HTML and Markdown output formats.

## Output
- HTML with inline styles (paste directly into Confluence)
- Markdown with Mermaid diagrams (Confluence Cloud compatible)
- Standalone Mermaid diagram files

## Sections Generated
1. Overview & Technology Stack
2. Architecture (Flowchart)
3. Data Flow (Sequence Diagram)
4. Source Schema (ER Diagram)
5. Medallion Architecture
6. Job Configuration
7. Data Quality
8. Operational Guide
9. Troubleshooting

In [ ]:
from pathlib import Path
import re
import yaml
from datetime import datetime
from dataclasses import dataclass
from typing import Any

## Configuration

In [ ]:
# Configuration
DOCS_OUTPUT_PATH = "/Workspace/cdc_lakehouse_docs"
REPO_CONFIG = {
    "source_schema_file": "/Workspace/init-db.sql",
    "job_config_file": "/Workspace/Orders-ingest-job.yaml"
}

## Data Classes

In [ ]:
@dataclass
class TableSchema:
    name: str
    columns: list[dict[str, Any]]
    primary_key: str | None = None
    foreign_keys: list[dict[str, str]] = None
    
    def __post_init__(self):
        if self.foreign_keys is None:
            self.foreign_keys = []


@dataclass
class JobTask:
    task_key: str
    depends_on: list[str]
    notebook_path: str | None = None
    dbt_command: str | None = None

## Schema Parser

In [ ]:
def collect_source_schema(sql_path: str) -> list[TableSchema]:
    """Parse PostgreSQL schema from init-db.sql"""
    content = Path(sql_path).read_text()
    tables = []
    current_table: dict[str, Any] = {}
    columns = []
    
    for line in content.splitlines():
        line = line.strip()
        if line.upper().startswith("CREATE TABLE"):
            match = re.search(r'CREATE TABLE.*?(\w+)\s*\(', line, re.IGNORECASE)
            if match:
                if current_table:
                    tables.append(TableSchema(
                        name=current_table["name"],
                        columns=columns.copy(),
                        primary_key=current_table.get("pk")
                    ))
                current_table = {"name": match.group(1)}
                columns = []
        elif line.upper().startswith("PRIMARY KEY"):
            match = re.search(r'PRIMARY KEY\s*\((.*?)\)', line, re.IGNORECASE)
            if match and current_table:
                current_table["pk"] = match.group(1)
        elif line.upper().startswith("FOREIGN KEY"):
            match = re.search(r'FOREIGN KEY\s*\((.*?)\)\s*REFERENCES\s+(\w+)\s*\((.*?)\)', line, re.IGNORECASE)
            if match and current_table:
                if "fks" not in current_table:
                    current_table["fks"] = []
                current_table["fks"].append({
                    "column": match.group(1),
                    "references": f"{match.group(2)}({match.group(3)})"
                })
        elif line and not line.startswith("DO $") and not line.startswith("BEGIN") and not line.startswith("END"):
            col_match = re.match(r'(\w+)\s+(\w+(?:\([^)]+\))?)', line)
            if col_match and current_table and not line.startswith(")"):
                col_name = col_match.group(1)
                col_type = col_match.group(2)
                is_pk = "PRIMARY KEY" in line.upper()
                is_not_null = "NOT NULL" in line.upper()
                default_match = re.search(r"DEFAULT\s+(\w+)", line, re.IGNORECASE)
                default = default_match.group(1) if default_match else None
                columns.append({
                    "name": col_name,
                    "type": col_type,
                    "nullable": not is_not_null,
                    "default": default,
                    "primary_key": is_pk
                })
    
    if current_table:
        tables.append(TableSchema(
            name=current_table["name"],
            columns=columns.copy(),
            primary_key=current_table.get("pk"),
            foreign_keys=current_table.get("fks", [])
        ))
    
    return tables

print("Schema parser loaded")

## Job Config Parser

In [ ]:
def parse_job_config(yaml_path: str) -> dict:
    """Parse Databricks job configuration from YAML"""
    content = Path(yaml_path).read_text()
    data = yaml.safe_load(content)
    
    job = data.get("resources", {}).get("jobs", {})
    job_name = list(job.keys())[0] if job else "unknown"
    job_data = job[job_name] if job else {}
    
    tasks = []
    for task in job_data.get("tasks", []):
        task_key = task.get("task_key", "")
        depends_on = [d.get("task_key") for d in task.get("depends_on", [])]
        
        notebook_task = task.get("notebook_task", {})
        notebook_path = notebook_task.get("notebook_path")
        
        dbt_task = task.get("dbt_task", {})
        dbt_command = dbt_task.get("commands", [""])[0] if dbt_task.get("commands") else None
        
        tasks.append({
            "task_key": task_key,
            "depends_on": depends_on,
            "notebook_path": notebook_path,
            "dbt_command": dbt_command
        })
    
    schedule_data = job_data.get("schedule", {})
    schedule = schedule_data.get("quartz_cron_expression") if schedule_data else None
    
    return {
        "name": job_name,
        "tasks": tasks,
        "schedule": schedule
    }

print("Job config parser loaded")

## Mermaid Diagram Generators

In [ ]:
def generate_architecture_diagram() -> str:
    """Generate architecture flowchart Mermaid diagram"""
    return '''```mermaid
flowchart LR
    subgraph Source["Source Systems"]
        PG[(PostgreSQL<br/>orders, products)]
    end
    
    subgraph CDC["CDC Layer"]
        DBZ[Debezium<br/>Connect]
        KB[Kafka<br/>cdc.public.*]
    end
    
    subgraph Databricks["Databricks Lakehouse"]
        subgraph Bronze["Bronze Layer"]
            BO[workspace.bronze<br/>orders]
            BP[workspace.bronze<br/>products]
        end
        
        subgraph Silver["Silver Layer"]
            SO[workspace.silver<br/>silver_orders]
            SP[workspace.silver<br/>silver_products]
        end
        
        subgraph Gold["Gold Layer"]
            GO[workspace.gold<br/>total_products_order]
        end
    end
    
    subgraph dbt["dbt Gold"]
        DBT[dbt-core<br/>dbt-databricks]
    end
    
    PG -->|WAL/CDC| DBZ
    DBZ -->|CDC Events| KB
    KB -->|Stream| BO
    KB -->|Stream| BP
    BO -->|Merge| SO
    BP -->|Merge| SP
    SO -->|Query| DBT
    SP -->|Query| DBT
    DBT -->|Build + Tests| GO
    
    style Source fill:#e3f2fd,stroke:#1976d2
    style CDC fill:#fff3e0,stroke:#f57c00
    style Bronze fill:#e8f5e9,stroke:#388e3c
    style Silver fill:#f3e5f5,stroke:#7b1fa2
    style Gold fill:#fce4ec,stroke:#c2185b
    style dbt fill:#fafafa,stroke:#616161
```'''

In [ ]:
def generate_dataflow_diagram() -> str:
    """Generate sequence diagram Mermaid diagram"""
    return '''```mermaid
sequenceDiagram
    participant P as PostgreSQL
    participant D as Debezium
    participant K as Kafka
    participant B as Bronze
    participant S as Silver
    participant G as Gold
    
    Note over P: INSERT/UPDATE/DELETE<br/>on orders/products
    
    P->>D: WAL Change Event
    D->>K: Serialize CDC Payload
    K->>B: Spark Structured Streaming
    
    loop Bronze Ingestion
        B->>B: Parse Debezium JSON
        B->>B: Write raw to Delta
    end
    
    B->>S: Read Bronze micro-batch
    
    loop Silver Merge
        S->>S: Deduplicate by key
        S->>S: Merge to current state
        S->>S: Schema evolution
    end
    
    S->>G: dbt source query
    
    loop Gold Build
        G->>G: Run dbt models
        G->>G: Execute DQ tests
    end
    
    Note over G: Total products<br/>by color
```'''

In [ ]:
def generate_er_diagram(tables: list[TableSchema]) -> str:
    """Generate ER diagram Mermaid diagram"""
    lines = ["```mermaid", "erDiagram"]
    
    for table in tables:
        lines.append(f"    {table.name.upper()} {{")
        for col in table.columns:
            pk_marker = "PK" if col.get("primary_key") else ""
            null_marker = "NULL" if col.get("nullable") else "NOT NULL"
            lines.append(f'        {col["name"]} {col["type"]} {pk_marker} {null_marker}')
        lines.append("    }")
    
    # Add relationship
    lines.append("    PRODUCTS ||--o{ ORDERS : \"product_id\"")
    lines.append("```")
    
    return "\n".join(lines)

In [ ]:
def generate_job_diagram(job: dict) -> str:
    """Generate job task dependency diagram"""
    lines = ["```mermaid", "graph TD"]
    
    for task in job.get("tasks", []):
        task_key = task["task_key"]
        if task.get("notebook_path"):
            label = task["notebook_path"].split("/")[-1].replace("_", " ")
            lines.append(f'    {task_key}["{label}"]')
        elif task.get("dbt_command"):
            label = task["dbt_command"].replace("--", " ").replace("_", " ")
            lines.append(f'    {task_key}["{label}"]')
    
    for task in job.get("tasks", []):
        for dep in task.get("depends_on", []):
            lines.append(f'    {dep} --> {task["task_key"]}')
    
    lines.append("```")
    
    return "\n".join(lines)

## HTML Generator

In [ ]:
def generate_html(title: str, sections: dict[str, str], timestamp: str) -> str:
    """Generate Confluence-compatible HTML"""
    
    html = f'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title}</title>
    <script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>
    <style>
        :root {{
            --primary-color: #0052cc;
            --secondary-color: #172b4d;
            --accent-color: #00875a;
            --bg-color: #ffffff;
            --border-color: #dfe1e6;
            --code-bg: #f4f5f7;
        }}
        * {{ box-sizing: border-box; margin: 0; padding: 0; }}
        body {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
            line-height: 1.6;
            color: var(--secondary-color);
            background: var(--bg-color);
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
        }}
        h1 {{ color: var(--primary-color); border-bottom: 3px solid var(--primary-color); padding-bottom: 10px; margin-bottom: 20px; }}
        h2 {{ color: var(--primary-color); border-left: 4px solid var(--accent-color); padding-left: 12px; margin: 30px 0 15px 0; }}
        h3 {{ color: var(--secondary-color); margin: 20px 0 10px 0; }}
        p {{ margin-bottom: 15px; }}
        code {{ background: var(--code-bg); padding: 2px 6px; border-radius: 3px; font-family: monospace; }}
        pre {{ background: var(--code-bg); padding: 15px; border-radius: 5px; overflow-x: auto; margin: 15px 0; border: 1px solid var(--border-color); }}
        table {{ width: 100%; border-collapse: collapse; margin: 20px 0; }}
        th, td {{ padding: 12px; text-align: left; border: 1px solid var(--border-color); }}
        th {{ background: var(--primary-color); color: white; }}
        tr:nth-child(even) {{ background: #f9f9f9; }}
        .toc {{ background: #f4f5f7; padding: 20px; border-radius: 5px; margin-bottom: 30px; }}
        .toc h3 {{ margin-top: 0; }}
        .toc ul {{ list-style: none; padding-left: 0; }}
        .toc li {{ margin: 8px 0; }}
        .toc a {{ color: var(--primary-color); text-decoration: none; }}
        .info-box {{ background: #e3f2fd; border-left: 4px solid #1976d2; padding: 15px; margin: 20px 0; }}
        .warning-box {{ background: #fff3e0; border-left: 4px solid #f57c00; padding: 15px; margin: 20px 0; }}
        .footer {{ margin-top: 50px; padding-top: 20px; border-top: 1px solid var(--border-color); font-size: 0.85em; color: #6b778c; text-align: center; }}
    </style>
</head>
<body>
    <h1>{title}</h1>
    
    <table>
        <tr><th>Property</th><th>Value</th></tr>
        <tr><td>Generated</td><td>{timestamp}</td></tr>
        <tr><td>Version</td><td>1.0.0</td></tr>
        <tr><td>Pipeline</td><td>CDC Lakehouse</td></tr>
    </table>
    
    <div class="toc">
        <h3>Table of Contents</h3>
        <ul>\n'''
    
    for i, section_title in enumerate(sections.keys(), 1):
        anchor = section_title.lower().replace(" ", "-")
        html += f'            <li><a href="#{anchor}">{i}. {section_title}</a></li>\n'
    
    html += '''        </ul>
    </div>\n'''
    
    for i, (section_title, section_content) in enumerate(sections.items(), 1):
        anchor = section_title.lower().replace(" ", "-")
        html += f'    <h2 id="{anchor}">{i}. {section_title}</h2>\n'
        html += f'    {section_content}\n'
    
    html += f'''
    <div class="footer">
        <p>Generated by Confluence Documentation Generator | CDC Lakehouse Lab</p>
    </div>
    
    <script>
        mermaid.initialize({{ 
            startOnLoad: true,
            theme: 'default',
            securityLevel: 'loose'
        }});
    </script>
</body>
</html>'''
    
    return html

## Markdown Generator

In [ ]:
def generate_markdown(title: str, sections: dict[str, str], timestamp: str) -> str:
    """Generate Confluence-compatible Markdown"""
    
    md = f"# {title}\n\n"
    md += f"| Property | Value |\n|----------|-------|\n"
    md += f"| Generated | {timestamp} |\n"
    md += f"| Version | 1.0.0 |\n"
    md += f"| Pipeline | CDC Lakehouse |\n\n"
    
    md += "## Table of Contents\n\n"
    for i, section_title in enumerate(sections.keys(), 1):
        anchor = section_title.lower().replace(" ", "-")
        md += f"{i}. [{section_title}](#{anchor})\n"
    
    md += "\n---\n\n"
    
    for i, (section_title, section_content) in enumerate(sections.items(), 1):
        anchor = section_title.lower().replace(" ", "-")
        md += f"## {i}. {section_title} {{#{anchor}}}\n\n"
        
        # Clean up HTML for Markdown
        content = section_content
        content = content.replace('<pre class="mermaid">', '```mermaid\n')
        content = content.replace('</pre>', '\n```')
        content = content.replace('<table>', '')
        content = content.replace('</table>', '')
        content = content.replace('<thead>', '')
        content = content.replace('</thead>', '')
        content = content.replace('<tbody>', '')
        content = content.replace('</tbody>', '')
        content = content.replace('<tr>', '|')
        content = content.replace('</tr>', '\n')
        content = content.replace('<th>', '|')
        content = content.replace('</th>', '|')
        content = content.replace('<td>', '|')
        content = content.replace('</td>', '|')
        content = content.replace('<br/>', '  ')
        content = content.replace('<strong>', '**')
        content = content.replace('</strong>', '**')
        content = content.replace('<div class="info-box">', '')
        content = content.replace('</div>', '')
        content = content.replace('<em>', '*')
        content = content.replace('</em>', '*')
        
        md += content + "\n\n"
    
    md += "---\n\n"
    md += "*Generated by Confluence Documentation Generator | CDC Lakehouse Lab*\n"
    
    return md

## Build Documentation

In [ ]:
# Build sections dictionary
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

tables = collect_source_schema(REPO_CONFIG["source_schema_file"])
job = parse_job_config(REPO_CONFIG["job_config_file"])

sections = {}

# Overview
sections["Overview"] = '''<div class="info-box">
This documentation describes the end-to-end CDC pipeline from PostgreSQL through Debezium and Kafka into Databricks Lakehouse with dbt Gold transformations.
</div>

### Technology Stack

| Component | Technology | Version |
|-----------|------------|----------|
| Source Database | PostgreSQL | 15 |
| CDC Capture | Debezium | 2.5 |
| Message Broker | Apache Kafka | 7.6.0 |
| Lakehouse Platform | Databricks | Latest |
| Gold Transformation | dbt | 1.x |
| Storage | Delta Lake | - |
| Catalog | Unity Catalog | - |'''

# Architecture
sections["Architecture"] = generate_architecture_diagram() + """

### Architecture Overview
The pipeline follows a medallion architecture pattern:
- **Bronze Layer**: Raw CDC events from Kafka
- **Silver Layer**: Current-state tables with deduplication
- **Gold Layer**: Business aggregates via dbt"""

# Data Flow
sections["Data Flow"] = generate_dataflow_diagram()

# Source Schema
sections["Source Schema"] = generate_er_diagram(tables) + "\n\n### Table Definitions\n\n"
for table in tables:
    sections["Source Schema"] += f"#### {table.name.upper()}\n\n"
    sections["Source Schema"] += "| Column | Type | Nullable | Key |\n"
    sections["Source Schema"] += "|--------|------|----------|-----|\n"
    for col in table.columns:
        key = "PK" if col.get("primary_key") else ""
        nullable = "YES" if col.get("nullable") else "NO"
        sections["Source Schema"] += f"| {col['name']} | {col['type']} | {nullable} | {key} |\n"
    sections["Source Schema"] += "\n"

# Medallion Architecture
sections["Medallion Architecture"] = """### Bronze Layer

| Schema | Table | Description |
|--------|-------|-------------|
| workspace.bronze | orders | Raw Debezium CDC events |
| workspace.bronze | products | Raw Debezium CDC events |

### Silver Layer

| Schema | Table | Description |
|--------|-------|-------------|
| workspace.silver | silver_orders | Current-state with schema evolution |
| workspace.silver | silver_products | Current-state with schema evolution |

### Gold Layer

| Schema | Table | Description |
|--------|-------|-------------|
| workspace.gold | total_products_order | Aggregate by product and color |

### Schema Evolution
The Silver layer supports schema evolution with configurable policies."""

# Job Configuration
sections["Job Configuration"] = f"""### Job: {job['name']}

| Property | Value |
|----------|-------|
| Schedule | {job.get('schedule') or 'Paused'} |

### Tasks
"""
for task in job.get("tasks", []):
    sections["Job Configuration"] += f"#### {task['task_key']}\n\n"
    if task.get("notebook_path"):
        sections["Job Configuration"] += f"- **Type**: Notebook\n- **Path**: `{task['notebook_path']}`\n"
    if task.get("dbt_command"):
        sections["Job Configuration"] += f"- **Type**: dbt Task\n- **Command**: `{task['dbt_command']}`\n"
    if task.get("depends_on"):
        sections["Job Configuration"] += f"- **Depends on**: {', '.join(task['depends_on'])}\n"
    sections["Job Configuration"] += "\n"

sections["Job Configuration"] += generate_job_diagram(job)

# Data Quality
sections["Data Quality"] = """### dbt Tests

- NOT NULL validation
- Uniqueness checks
- Business rule validation

### Schema Drift Detection

| Policy | Behavior |
|--------|----------|
| strict | Block any schema change |
| additive_only | Allow new columns only (default) |
| permissive | Log drift but never block |"""

# Operational Guide
sections["Operational Guide"] = """### Starting the Pipeline

```bash
docker compose up -d
curl -X POST http://localhost:8083/connectors -H 'Content-Type: application/json' --data @postgres-connector.json
python3 generators/load_products_generator.py
python3 generators/load_generator.py
```

### Environment Variables

| Variable | Description |
|----------|-------------|
| DATABRICKS_HOST | Databricks workspace URL |
| DATABRICKS_TOKEN | API authentication token |
| KAFKA_BOOTSTRAP | Kafka connection (ngrok) |"""

# Troubleshooting
sections["Troubleshooting"] = """### Common Issues

| Issue | Cause | Resolution |
|-------|-------|------------|
| dbt command not found | Missing dependencies | Install dbt packages |
| Kafka connection timeout | ngrok tunnel expired | Restart ngrok |
| Schema drift errors | Source schema changed | Update Silver metadata |

### Debug Queries

```sql
-- Check Bronze
SELECT * FROM workspace.bronze.orders LIMIT 10;

-- Check Silver
SELECT id, COUNT(*) as cnt FROM workspace.silver.silver_orders GROUP BY id HAVING cnt > 1;

-- Verify Gold
SELECT * FROM workspace.gold.total_products_order;
```"""

print(f"Built {len(sections)} documentation sections")

## Generate Output Files

In [ ]:
import os

# Create output directory
output_path = Path(DOCS_OUTPUT_PATH)
output_path.mkdir(parents=True, exist_ok=True)

# Generate outputs
html_content = generate_html("CDC Lakehouse Pipeline Documentation", sections, timestamp)
md_content = generate_markdown("CDC Lakehouse Pipeline Documentation", sections, timestamp)

# Write files
html_path = output_path / "confluence_html.html"
md_path = output_path / "confluence_markdown.md"

html_path.write_text(html_content)
md_path.write_text(md_content)

print(f"Generated:")
print(f"  - {html_path}")
print(f"  - {md_path}")

## Display Markdown Output

In [ ]:
display(markdown(md_content))